# 02 · The zero-shot baseline - the number the fine-tune has to beat

Before training anything, score the *unadapted* base model on exactly the rows the adapter will be scored
on later. Without this number, "the fine-tuned model gets 71%" means nothing: maybe the base already got
70%, maybe it got 30%. The difference between this notebook and `04_evaluate_adapter` is the entire
justification for the training run.

**How to read the scores.** Each answer type is scored the way its benchmark is scored - exact match
for yes/no and letters, IoU ≥ 0.5 for boxes, ROUGE-L for captions - and *overall* is the mean over
types, not over rows, so 1,000 yes/no rows cannot hide 20 boxes. Chance is 50% for yes/no and 25% for
MCQ; a base model near chance on MCQ is not broken, it has simply never seen a CORINE class name.

**Mean stated confidence** is the model's average token probability. Compare it to accuracy: a model at
0.9 confidence and 0.55 accuracy is over-confident, and S18's aggregation will carry that number into
the answer's stated confidence - one reason the adapter is expected to help *calibration*, not only
accuracy.

Runs on the laptop (2B, 4-bit, ~3 s per row). `VLM_ADAPTER_REPOSITORY` must be unset.

In [ ]:
import sys, pathlib, asyncio, json
BACKEND = pathlib.Path.cwd().resolve()
while BACKEND.name != "backend":
    BACKEND = BACKEND.parent
sys.path.insert(0, str(BACKEND))
import os; os.chdir(BACKEND)
import pandas as pd
from app.config import settings
from app.models.manager import get_manager, reset_manager
from app.services.evaluation.vqa import evaluate_vqa

# The same three files and limits for the base and for the adapter, so rows pair up exactly.
FILES = {
    "bigearthnet_txt.bench": (BACKEND / "data/training/vlm/bigearthnet_txt.bench.jsonl", None),   # 189 human-verified
    "bigearthnet_txt.test": (BACKEND / "data/training/vlm/bigearthnet_txt.test.jsonl", 400),     # template rows
    "rsvqa_lr.test": (BACKEND / "data/training/vlm/rsvqa_lr.test.jsonl", 550),                    # another country
}
PREDICTIONS = BACKEND / "data/training/vlm/predictions"

async def score(tag):
    manager = await get_manager()
    reports = {}
    for name, (file, limit) in FILES.items():
        if not file.exists():
            print("missing", file); continue
        reports[name] = await evaluate_vqa(manager=manager, file=file, limit=limit,
                                           predictions_path=PREDICTIONS / f"{tag}.{name}.jsonl")
    await reset_manager()
    return reports

def table(reports):
    rows = []
    for name, report in reports.items():
        for kind in sorted(report.score.asked):
            rows.append({"file": name, "type": kind, "n": report.score.asked[kind],
                         "score": round(report.score.accuracy(kind), 3), "model": report.model_version})
        rows.append({"file": name, "type": "OVERALL (mean of types)", "n": report.samples,
                     "score": round(report.score.overall, 3), "model": report.model_version})
    return pd.DataFrame(rows)

In [ ]:
assert settings.vlm_adapter_repository is None, "unset VLM_ADAPTER_REPOSITORY for the baseline"
reports = await score("base")
baseline = table(reports)
baseline.to_csv(BACKEND / "data/training/vlm/scores_baseline.csv", index=False)
baseline

## What the base model actually says

Accuracy is a summary; the examples are the diagnosis. Look for: answers in the wrong *format* (a
sentence where a letter was asked for - a format problem the LoRA fixes in the first hundred steps),
boxes that are plausible but shifted (a grounding problem), and captions that describe a generic
"aerial view" (a domain-vocabulary problem: the model does not know what "non-irrigated arable land"
looks like at 10 m).

In [ ]:
for name, report in reports.items():
    print(f"== {name}  mean stated confidence {report.mean_confidence:.2f}")
    for kind, prompt, reference, prediction, hit in report.examples[:6]:
        print(f"  [{'ok ' if hit else 'BAD'}] {kind:<13} Q: {prompt[:80]}")
        print(f"                     ref: {reference[:60]!r}   got: {prediction[:60]!r}")